# Running the MCMC chains on current data

The current-data constraints in Section III of Cheslog et al. (2026) come
from twelve Cobaya runs. Five of them use a power-law primordial spectrum
and give Table III and Figure 1. The other seven replace the power law with
an amplitude in each of a set of $k$ bins and give Figure 3, Figures 10 and
11, and Table VIII.

This notebook goes through the whole path. It shows what the input files
contain, what has to be installed to run them, how to fill in the paths, the
exact command for each run, and how to turn the finished chains into the
numbers the plotting notebook reads. At the end, `hdInitPk_plots.ipynb` can
be pointed at your own chains instead of the thinned ones distributed with
the package.

## How the chains in the paper were run

We ran every chain on the SeaWulf computing system at Stony Brook
University, which is maintained by Stony Brook Research Computing and
Cyberinfrastructure and the Institute for Advanced Computational Science
(see the acknowledgments of the paper). Each run used six MPI processes, so
each run is a set of six Metropolis-Hastings chains that Cobaya compares to
decide when to stop. As described in Section III of the paper, we remove the
first half of every chain as burn-in and consider a run converged once the
Gelman-Rubin statistic $R-1$ is below 0.01.

These are long runs. Expect days on a cluster, and longer for the binned
$\mathcal{P}(k)$ chains, which sample up to 58 parameters and evaluate CAMB
at the accuracy settings of Appendix A at every step. Nothing here needs to
be run in one sitting. Every input file sets `resume: true`, so a run that
stops can be started again with the same command and picks up where it left
off.

## The input files

They ship with the package, in `hdinitpk/cobaya_yaml_files`.

In [13]:
import os
import glob

import numpy as np
import pandas as pd
import yaml

from getdist import mcsamples
import hdinitpk

# the cobaya inputs sit beside the data directory in the package
yaml_dir = os.path.normpath(os.path.join(os.path.dirname(hdinitpk.DATA_DIR),
                                         'cobaya_yaml_files'))

for sub in ('current_data', 'binned_pk', 'HD'):
    files = sorted(f for f in os.listdir(os.path.join(yaml_dir, sub))
                   if f.endswith('.yaml'))
    print(f'{sub}/  ({len(files)} files)')
    for f in files:
        print(f'    {f}')

current_data/  (5 files)
    lcdm_nrun.yaml
    lcdm_nrun_nnu.yaml
    lcdm_nrun_nnu_mnu.yaml
    p_act_lb.yaml
    w0wacdm_nrun.yaml
binned_pk/  (7 files)
    cmb_pas_30_bins_arbitrary_binning.yaml
    cmb_pas_7_bins_arbitrary_binning.yaml
    cmb_pas_desi_30_bins_arbitrary_binning.yaml
    cmb_pas_desi_7_bins_arbitrary_binning.yaml
    p_act_lb_30_bins_arbitrary_binning.yaml
    p_act_lb_30_bins_no_sroll_arbitrary_binning.yaml
    p_act_lb_7_bins_no_sroll_arbitrary_binning.yaml
HD/  (4 files)
    camb_lcdm_likelihood.yaml
    camb_nrun_likelihood.yaml
    class_lcdm_likelihood.yaml
    class_nrun_likelihood.yaml


The five files in `current_data` are the power-law runs. Four use CMB-PAS
with DESI DR2, adding one parameter at a time. The fifth is the P-ACT-LB
comparison column of Table III.

| File | Model | Data |
|---|---|---|
| `lcdm_nrun.yaml` | $\Lambda$CDM + $\alpha_s$ | CMB-PAS + DESI DR2 |
| `w0wacdm_nrun.yaml` | $w_0w_a$CDM + $\alpha_s$ | CMB-PAS + DESI DR2 |
| `lcdm_nrun_nnu.yaml` | $\Lambda$CDM + $\alpha_s$ + $N_\mathrm{eff}$ | CMB-PAS + DESI DR2 |
| `lcdm_nrun_nnu_mnu.yaml` | $\Lambda$CDM + $\alpha_s$ + $N_\mathrm{eff}$ + $\sum m_\nu$ | CMB-PAS + DESI DR2 |
| `p_act_lb.yaml` | $\Lambda$CDM + $\alpha_s$ | P-ACT-LB |

The seven files in `binned_pk` sample an amplitude in each $k$ bin in place
of $A_s$, $n_s$ and $\alpha_s$. They use the `BinnedPk` theory class in
`arbitrary_Pkbinning.py` at the root of this repository, with the bins set
to the seven- and 30-bin schemes of the paper.

| File | Data | Bins |
|---|---|---|
| `cmb_pas_7_bins_arbitrary_binning.yaml` | CMB-PAS | 7 |
| `cmb_pas_desi_7_bins_arbitrary_binning.yaml` | CMB-PAS + DESI DR2 | 7 |
| `p_act_lb_7_bins_no_sroll_arbitrary_binning.yaml` | P-ACT-LB, without `sroll2` | 7 |
| `cmb_pas_30_bins_arbitrary_binning.yaml` | CMB-PAS | 30 |
| `cmb_pas_desi_30_bins_arbitrary_binning.yaml` | CMB-PAS + DESI DR2 | 30 |
| `p_act_lb_30_bins_arbitrary_binning.yaml` | P-ACT-LB | 30 |
| `p_act_lb_30_bins_no_sroll_arbitrary_binning.yaml` | P-ACT-LB, without `sroll2` | 30 |

The four files in `HD` are the CMB-HD mock chains of Figure 9, which check
the Fisher forecasts against a full MCMC. They use the hdlike likelihood
rather than current data, and are covered at the end of the section on
running the chains.

In [2]:
def sampled_params(info):
    """Parameter names Cobaya will sample, meaning the ones with a prior."""
    out = []
    for name, spec in (info.get('params') or {}).items():
        if isinstance(spec, dict) and 'prior' in spec:
            out.append(name)
    for lik in (info.get('likelihood') or {}).values():
        if isinstance(lik, dict):
            for name, spec in (lik.get('params') or {}).items():
                if isinstance(spec, dict) and 'prior' in spec:
                    out.append(name)
    return out


class Loose(yaml.SafeLoader):
    pass

# the candl and clipy blocks use !!python/name: tags. We only want to read
# the file, not import anything.
Loose.add_multi_constructor('!!python/name:', lambda l, s, n: s)
Loose.add_multi_constructor('tag:yaml.org,2002:python/name:',
                            lambda l, s, n: s)


def read_input(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.load(f.read(), Loader=Loose)


rows = []
for sub in ('current_data', 'binned_pk'):
    for path in sorted(glob.glob(os.path.join(yaml_dir, sub, '*.yaml'))):
        info = read_input(path)
        theory = list((info.get('theory') or {}))
        rows.append({
            'file': f'{sub}/{os.path.basename(path)}',
            'theory': ', '.join(theory),
            'likelihoods': len(info.get('likelihood') or {}),
            'sampled': len(sampled_params(info)),
        })
pd.DataFrame(rows).set_index('file')

,theory,likelihoods,sampled
file,,,
current_data/lcdm_nrun.yaml,camb,8,33
current_data/lcdm_nrun_nnu.yaml,camb,8,34
current_data/lcdm_nrun_nnu_mnu.yaml,camb,8,35
current_data/p_act_lb.yaml,camb,6,9
current_data/w0wacdm_nrun.yaml,camb,8,35
binned_pk/cmb_pas_30_bins_arbitrary_binning.yaml,"arbitrary_Pkbinning.BinnedPk, camb",7,60
binned_pk/cmb_pas_7_bins_arbitrary_binning.yaml,"arbitrary_Pkbinning.BinnedPk, camb",7,37
binned_pk/cmb_pas_desi_30_bins_arbitrary_binning.yaml,"arbitrary_Pkbinning.BinnedPk, camb",8,60
binned_pk/cmb_pas_desi_7_bins_arbitrary_binning.yaml,"arbitrary_Pkbinning.BinnedPk, camb",8,37


## What you need installed

None of these likelihoods are dependencies of `hdinitpk`. Install the ones
the runs you want require. The *Planck* low-$\ell$ and DESI BAO likelihoods
are built into Cobaya and only need `cobaya-install` to fetch their data.

| In the input file | What it is | Where to get it |
|---|---|---|
| `clipy_lowl_tt`, `clipy_highl_*` | *Planck* 2018 PR3, through the pure-Python `clik` reimplementation | [benabed/clipy](https://github.com/benabed/clipy), with the data from the [Planck Legacy Archive](https://pla.esac.esa.int/) |
| `candl_like` | SPT-3G D1 T&E | [Lbalkenhol/candl](https://github.com/Lbalkenhol/candl), with the data in [SouthPoleTelescope/spt_candl_data](https://github.com/SouthPoleTelescope/spt_candl_data) |
| `act_dr6_cmbonly.*` | ACT DR6 CMB-only (ACT-lite) | [ACTCollaboration/DR6-ACT-lite](https://github.com/ACTCollaboration/DR6-ACT-lite) |
| `act_dr6_lenslike.ACTDR6LensLike` | ACT DR6 + *Planck* lensing | [ACTCollaboration/act_dr6_lenslike](https://github.com/ACTCollaboration/act_dr6_lenslike) |
| `act_dr6_spt_lenslike.ACTDR6LensLike` | joint ACT + *Planck* + SPT-3G lensing | [qujia7/spt_act_likelihood](https://github.com/qujia7/spt_act_likelihood) |
| `planck_2018_lowl.*`, `bao.desi_*` | *Planck* low-$\ell$, DESI BAO | built into Cobaya |
| `hdlike.hdlike.HDLike` | the CMB-HD mock likelihood, for the `HD` files only | [CMB-HD/hdlike](https://github.com/CMB-HD/hdlike) |

Two theory-side requirements are easy to miss. Every current-data file sets
`recombination_model: CosmoRec`, so CAMB has to be built against
[CosmoRec](https://www.jb.man.ac.uk/~jchluba/Science/CosmoRec/CosmoRec.html),
which is not the default. The binned $\mathcal{P}(k)$ runs also need
`external_primordial_pk: true` on the CAMB block, which is already set in
the shipped files.

## Filling in the paths

Every absolute path in the shipped files is written as a `/path/to/...`
placeholder. The cell below lists what each file needs.

In [3]:
def placeholders(path):
    found = set()
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            for token in line.split():
                if token.startswith('/path/to/'):
                    found.add('/'.join(token.split('/')[:4]))
    return sorted(found)


for sub in ('current_data', 'binned_pk', 'HD'):
    for path in sorted(glob.glob(os.path.join(yaml_dir, sub, '*.yaml'))):
        p = placeholders(path)
        if p:
            print(f'{sub}/{os.path.basename(path):50s} {", ".join(p)}')

current_data/lcdm_nrun.yaml                                     /path/to/camb, /path/to/chains, /path/to/hdInitPk, /path/to/planck_data
current_data/lcdm_nrun_nnu.yaml                                 /path/to/camb, /path/to/chains, /path/to/hdInitPk, /path/to/planck_data
current_data/lcdm_nrun_nnu_mnu.yaml                             /path/to/camb, /path/to/chains, /path/to/hdInitPk, /path/to/planck_data
current_data/p_act_lb.yaml                                      /path/to/camb, /path/to/chains, /path/to/hdInitPk
current_data/w0wacdm_nrun.yaml                                  /path/to/camb, /path/to/chains, /path/to/hdInitPk, /path/to/planck_data
binned_pk/cmb_pas_30_bins_arbitrary_binning.yaml             /path/to/camb, /path/to/chains, /path/to/hdInitPk, /path/to/hdlike, /path/to/planck_data, /path/to/spt_candl_data
binned_pk/cmb_pas_7_bins_arbitrary_binning.yaml              /path/to/chains, /path/to/hdInitPk, /path/to/hdlike, /path/to/planck_data, /path/to/spt_candl_data
binned_

`/path/to/hdInitPk` is this repository. Cobaya needs it to import
`arbitrary_Pkbinning` and to find the proposal matrices in
`hdinitpk/data/proposal_matrices`. `/path/to/chains` is where the chains
are written. `/path/to/hdMockData` is your hdMockData checkout, for the
BBN table the CLASS runs read. The rest point at the external likelihood
data.

Rather than editing the shipped files, the cell below copies each one into
an `mcmc_runs` directory next to this notebook and fills in the
placeholders. It also sets the `output` line of each copy so that the chain
is written to `hdinitpk/data/user_generated_data/chains` under the name of
its input file. That is where the plotting notebook and
`run_hdInitPk_to_LinearPk.py` look for chains you have run yourself.

In [4]:
run_dir = os.path.join(os.getcwd(), 'mcmc_runs')
chain_dir = hdinitpk.user_data_path('chains')

substitutions = {
    '/path/to/hdInitPk': os.path.normpath(
        os.path.join(os.path.dirname(hdinitpk.DATA_DIR), '..')),
    '/path/to/chains': chain_dir,
    # fill these in for the runs you want
    '/path/to/planck_data': '/your/planck/plc_3.0',
    '/path/to/spt_candl_data': '/your/spt_candl_data',
    '/path/to/hdlike': '/your/hdlike',
    '/path/to/hdMockData': '/your/hdMockData',
    # p_act_lb.yaml points `path:` at a local CAMB build. Comment that
    # line out in the copy to use the installed CAMB instead.
    '/path/to/camb': '/your/camb',
}

os.makedirs(run_dir, exist_ok=True)


def prepare(sub, name):
    """Copy one input file into run_dir with the placeholders filled in,
    and with its chain written to chain_dir under the input file's name."""
    src = os.path.join(yaml_dir, sub, name)
    dst = os.path.join(run_dir, name)
    stem = os.path.splitext(name)[0]
    with open(src, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    out = []
    for line in lines:
        if line.startswith('output:'):
            line = f'output: {os.path.join(chain_dir, stem)}\n'
        for placeholder, value in substitutions.items():
            line = line.replace(placeholder, value)
        out.append(line)
    with open(dst, 'w', encoding='utf-8') as f:
        f.writelines(out)
    return dst

## Running the chains

Each run is six chains, one per MPI process, which is how every chain in the
paper was run. Once the copies in `mcmc_runs` are prepared, the twelve
commands are

```
mpirun -np 6 cobaya-run mcmc_runs/lcdm_nrun.yaml
mpirun -np 6 cobaya-run mcmc_runs/w0wacdm_nrun.yaml
mpirun -np 6 cobaya-run mcmc_runs/lcdm_nrun_nnu.yaml
mpirun -np 6 cobaya-run mcmc_runs/lcdm_nrun_nnu_mnu.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_7_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_desi_7_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb_7_bins_no_sroll_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_30_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_desi_30_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb_30_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb_30_bins_no_sroll_arbitrary_binning.yaml
```

A run that stops picks up where it left off when started again with the
same command. Add `--force` to discard the existing chain and start over.

Every input file points at a proposal covariance in
`hdinitpk/data/proposal_matrices`, built from that run's own converged
chain. That is what makes these runs tractable to restart. A new model with
no proposal takes much longer to burn in.

The cell below prepares all twelve copies and prints the same list.

In [5]:
# The chains this notebook loads afterwards, keyed by the name the plotting
# notebook uses, with the input file each one comes from.
power_law_runs = {
    'pas_lcdm_nrun':         'lcdm_nrun',
    'pas_w0wacdm_nrun':      'w0wacdm_nrun',
    'pas_lcdm_nrun_nnu':     'lcdm_nrun_nnu',
    'pas_lcdm_nrun_nnu_mnu': 'lcdm_nrun_nnu_mnu',
    'pact_lb_nrun':          'p_act_lb',
}

binned_pk_runs = {
    'binned_pk/pas_7bin':       'cmb_pas_7_bins_arbitrary_binning',
    'binned_pk/pas_desi_7bin':  'cmb_pas_desi_7_bins_arbitrary_binning',
    'binned_pk/pact_lb_7bin':   'p_act_lb_7_bins_no_sroll_arbitrary_binning',
    'binned_pk/pas_30bin':      'cmb_pas_30_bins_arbitrary_binning',
    'binned_pk/pas_desi_30bin': 'cmb_pas_desi_30_bins_arbitrary_binning',
    'binned_pk/pact_lb_30bin':  'p_act_lb_30_bins_arbitrary_binning',
}

all_runs = {**power_law_runs, **binned_pk_runs}

for name, yaml_name in all_runs.items():
    sub = 'binned_pk' if name.startswith('binned_pk/') else 'current_data'
    prepare(sub, f'{yaml_name}.yaml')
    print(f'mpirun -np 6 cobaya-run mcmc_runs/{yaml_name}.yaml')

# The one binned run the paper does not use downstream, but which is
# provided: P-ACT-LB with 30 bins and the sroll2 low-ell EE likelihood.
prepare('binned_pk', 'p_act_lb_30_bins_no_sroll_arbitrary_binning.yaml')
print('mpirun -np 6 cobaya-run '
      'mcmc_runs/p_act_lb_30_bins_no_sroll_arbitrary_binning.yaml')

mpirun -np 6 cobaya-run mcmc_runs/lcdm_nrun.yaml
mpirun -np 6 cobaya-run mcmc_runs/w0wacdm_nrun.yaml
mpirun -np 6 cobaya-run mcmc_runs/lcdm_nrun_nnu.yaml
mpirun -np 6 cobaya-run mcmc_runs/lcdm_nrun_nnu_mnu.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_7_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_desi_7_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb_7_bins_no_sroll_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_30_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/cmb_pas_desi_30_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb_30_bins_arbitrary_binning.yaml
mpirun -np 6 cobaya-run mcmc_runs/p_act_lb_30_bins_no_sroll_arbitrary_binning.yaml


### The CMB-HD mock chains

The four files in `HD` are run the same way, after their placeholders are
filled in. They need [hdlike](https://github.com/CMB-HD/hdlike), and the
two `class_*` files also need CLASS and its Python wrapper. The CLASS
files point at the BBN table in your hdMockData checkout, and set
`use_class: True` so that hdlike compares the theory to the CMB-HD
bandpowers calculated with CLASS.

```
mpirun -np 6 cobaya-run hdinitpk/cobaya_yaml_files/HD/camb_lcdm_likelihood.yaml
mpirun -np 6 cobaya-run hdinitpk/cobaya_yaml_files/HD/camb_nrun_likelihood.yaml
mpirun -np 6 cobaya-run hdinitpk/cobaya_yaml_files/HD/class_lcdm_likelihood.yaml
mpirun -np 6 cobaya-run hdinitpk/cobaya_yaml_files/HD/class_nrun_likelihood.yaml
```

These are not loaded by the rest of this notebook. The plotting notebook
reads the two nine-parameter chains for Figure 9.

## Loading the finished chains

Each chain is read from the `output` root set above. The burn-in is removed
on load, and `ignore_rows = 0.5` is what we used throughout the paper.

In [6]:
ignore_rows = 0.5

chain_roots = {name: os.path.join(chain_dir, yaml_name)
               for name, yaml_name in all_runs.items()}


def load_chain(root):
    return mcsamples.loadMCSamples(root, settings={'ignore_rows': ignore_rows})


chains = {}
for name, root in chain_roots.items():
    try:
        chains[name] = load_chain(root)
        print(f'  {name}')
    except Exception as err:
        print(f'  {name}: {type(err).__name__}: {err}')

print(f'\n{len(chains)} chains loaded')

  pas_lcdm_nrun: OSError: No chains found: /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains/lcdm_nrun
  pas_w0wacdm_nrun: OSError: No chains found: /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains/w0wacdm_nrun
  pas_lcdm_nrun_nnu: OSError: No chains found: /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains/lcdm_nrun_nnu
  pas_lcdm_nrun_nnu_mnu: OSError: No chains found: /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains/lcdm_nrun_nnu_mnu
  pact_lb_nrun: OSError: No chains found: /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains/p_act_lb
  binned_pk/pas_7bin: OSError: No chains found: /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains/cmb_pas_7_bins_ar

## Checking convergence

The paper's criterion is $R-1 < 0.01$ after the burn-in is removed. The
statistic compares the six chains a run produced, so it has to be computed
here, from the full chains, before they are merged or thinned. getdist
returns `None` rather than raising when it only found a single chain file,
so that case is handled explicitly below.

For the binned $\mathcal{P}(k)$ runs we compute the statistic ourselves,
following Section III of the paper. getdist requires the within-chain
covariance $W$ to be positive definite. With up to 58 sampled parameters,
several of them prior-dominated high-$k$ bin amplitudes, it often is not,
and getdist then reports nothing at all. The routine below uses the
pseudo-inverse of $W$ instead (`np.linalg.pinv`), which is what the paper
did.

In [7]:
def gelman_rubin_pinv(samples, nparam=None):
    """Multivariate Gelman-Rubin R-1 with a pseudo-inverse of the
    within-chain covariance, as in Section III of the paper.

    R-1 is (m+1)/m times the largest eigenvalue of pinv(W) @ B_over_N for
    m chains, where W is the mean of the per-chain covariances and B_over_N
    is the covariance of the per-chain means. Only the sampled parameters
    are used. Returns nan if the samples came from a single chain file.
    """
    try:
        chain_parts = samples.getSeparateChains()
    except Exception:
        return float('nan')
    m = len(chain_parts)
    if m < 2:
        return float('nan')

    if nparam is None:
        nparam = samples.paramNames.numNonDerived()
    n_common = min(part.samples.shape[0] for part in chain_parts)

    chain_means, chain_covs = [], []
    for part in chain_parts:
        s = part.samples[:n_common, :nparam]
        w = part.weights[:n_common]
        chain_means.append(np.average(s, weights=w, axis=0))
        chain_covs.append(np.cov(s.T, aweights=w, ddof=1))

    W = np.mean(chain_covs, axis=0)
    B_over_N = np.cov(np.array(chain_means), rowvar=False, ddof=1)
    evals = np.linalg.eigvals(np.linalg.pinv(W) @ B_over_N)
    return float(((m + 1) / m) * np.max(evals.real))


convergence = {}
for name, chain in chains.items():
    if name in binned_pk_runs:
        r = gelman_rubin_pinv(chain)
    else:
        r = chain.getGelmanRubin()
        r = float('nan') if r is None else float(r)
    convergence[name] = r

pd.Series(convergence, name='R-1').to_frame()

,R-1


## Table III, the current-constraints table

Marginalized means and $1\sigma$ errors for the five power-law chains. The
same numbers set the contours in Figure 1.

In [8]:
table_params = ['ombh2', 'omch2', 'H0', 'tau', 'logA', 'ns', 'nrun',
                'nnu', 'mnu', 'w', 'wa', 'sigma8', 'omegam']


def marge(chain, params):
    stats = chain.getMargeStats()
    out = {}
    for p in params:
        par = stats.parWithName(p)
        if par is not None and par.err != 0:
            out[p] = (par.mean, par.err)
    return out


rows = {}
for name in power_law_runs:
    if name not in chains:
        continue
    m = marge(chains[name], table_params)
    rows[name] = {p: f'{mean:.5g} \u00b1 {err:.2g}' for p, (mean, err) in m.items()}

pd.DataFrame(rows).T

""


## Comparing with the packaged chains

The 1-sigma errors of your chains next to those of the thinned chains
distributed with the package, and their ratio, one table per run. The two
should agree to within the sampling noise of the chains. Differences of a
few percent are normal, and the packaged chains are thinned, which adds a
little noise of its own.

In [9]:
# Compare the errors of your chains with those of the packaged chains, one
# table per run: your 1-sigma error, the packaged one, and their ratio.
from hdinitpk.plotting_utilities import print_table

packaged_chain_dir = hdinitpk.data_path('chains')


def packaged_chain(name):
    """The packaged chain of this name, found in any subdirectory of
    hdinitpk/data/chains. The packaged chains are thinned with the burn-in
    already removed, so nothing more is removed here."""
    roots = glob.glob(os.path.join(packaged_chain_dir, '*', f'{name}.txt'))
    if not roots:
        raise FileNotFoundError(f'no packaged chain {name} under {packaged_chain_dir}')
    return mcsamples.loadMCSamples(roots[0][:-4], settings={'ignore_rows': 0})


for name in power_law_runs:
    if name not in chains:
        continue
    yours = {p: err for p, (mean, err) in marge(chains[name], table_params).items()}
    packaged = {p: err for p, (mean, err) in marge(packaged_chain(name), table_params).items()}
    ratios = {p: yours[p] / packaged[p] for p in yours if p in packaged}
    print_table([yours, packaged, ratios],
                ['your chain', 'packaged chain', 'ratio (yours / packaged)'],
                title=f'{name}: 1-sigma errors',
                list_of_keys=[p for p in table_params if p in yours])

## The binned P(k) statistics

Figures 3, 10 and 11 use marginalized statistics of the bin amplitudes
rather than the chains themselves, so those are computed once here and
cached. The runs sample $b_n = 10^9 e^{-2\tau}\mathcal{P}_n(k)$. We also
want $\mathcal{P}_n(k)$ itself, which means multiplying back by $e^{2\tau}$
sample by sample.

In [10]:
def add_raw_pk(chain, nbins):
    """Add Pk1..PkN (the primordial amplitudes) to a chain that samples
    the e^{-2 tau}-scaled b1..bN."""
    pars = chain.getParams()
    tau_vec = getattr(pars, 'tau')
    existing = chain.getParamNames().list()
    for i in range(nbins):
        name = f'Pk{i+1}'
        if name in existing:
            continue
        amplitude = getattr(pars, f'b{i+1}')
        chain.addDerived(amplitude * np.exp(2.0 * tau_vec),
                         name=name, label=f'P(k_{{{i+1}}})')
    chain.updateBaseStatistics()


def binned_pk_stats(chain, params):
    """Mean, asymmetric 68% errors and symmetric error per bin, times 10^9."""
    stats = chain.getMargeStats()
    means, lower, upper, errs = [], [], [], []
    for p in params:
        s = stats.parWithName(p)
        mean = 1e9 * s.mean
        means.append(mean)
        lower.append(mean - 1e9 * s.limits[0].lower)
        upper.append(1e9 * s.limits[0].upper - mean)
        errs.append(1e9 * s.err)
    return means, lower, upper, errs

In [11]:
def collect_stats(runs, nbins):
    results = {}
    for name, tag in runs:
        if name not in chains:
            continue
        chain = chains[name]
        add_raw_pk(chain, nbins)
        for prefix, params in ((tag, [f'b{i+1}' for i in range(nbins)]),
                               (f'{tag}_pk', [f'Pk{i+1}' for i in range(nbins)])):
            means, lo, hi, err = binned_pk_stats(chain, params)
            results[f'{prefix}_means'] = means
            results[f'{prefix}_lower_errs'] = lo
            results[f'{prefix}_upper_errs'] = hi
            results[f'{prefix}_tot_err'] = err
    return results


stats_7bin = collect_stats([('binned_pk/pact_lb_7bin', 'p_act'),
                            ('binned_pk/pas_desi_7bin', 'pas_desi'),
                            ('binned_pk/pas_7bin', 'pas')], 7)

stats_30bin = collect_stats([('binned_pk/pact_lb_30bin', 'p_act'),
                             ('binned_pk/pas_desi_30bin', 'pas_desi'),
                             ('binned_pk/pas_30bin', 'pas')], 30)

print(f'{len(stats_7bin)} rows for the 7-bin scheme, '
      f'{len(stats_30bin)} for the 30-bin')

0 rows for the 7-bin scheme, 0 for the 30-bin


## Saving the results for the plotting notebook

The chains stay where Cobaya wrote them, in
`hdinitpk/data/user_generated_data/chains`, at full length and in Cobaya's
own format. The plotting notebook reads them from there directly and
removes the burn-in on load.

Two things computed from them are written out, since the plotting notebook
does not recompute them. The binned $\mathcal{P}(k)$ statistics above go to
`user_generated_data/cached_results`, and the $R-1$ values go next to the
chains, since they cannot be recovered once getdist has merged a chain.

In [12]:
cache_out = hdinitpk.user_data_path('cached_results')
convergence_file = os.path.join(chain_dir, 'chain_convergence.txt')


def write_results(path, title, results):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f'# {title}\n')
        f.write('# one row per quantity: the name, then its values\n')
        for key, values in results.items():
            row = ' '.join(repr(float(v)) for v in np.asarray(values).ravel())
            f.write(f'{key} {row}\n')
    print(f'  {path}')


# Keyed by the cobaya root name, which is the input file's name. The
# plotting notebook maps the names it uses onto these.
with open(convergence_file, 'w') as f:
    f.write('# Gelman-Rubin R-1 of each chain\n')
    f.write('# power-law runs: getdist getGelmanRubin()\n')
    f.write('# binned_pk runs: gelman_rubin_pinv(), the same statistic with a\n')
    f.write('#   pseudo-inverse of W and a factor (m+1)/m\n')
    for name, r in convergence.items():
        f.write(f'{os.path.basename(chain_roots[name])} {r!r}\n')
print(f'  {convergence_file}')

if stats_7bin:
    write_results(os.path.join(cache_out, 'binned_pk_stats_7bin.txt'),
                  'binned_pk_stats_7bin', stats_7bin)
if stats_30bin:
    write_results(os.path.join(cache_out, 'binned_pk_stats_30bin.txt'),
                  'binned_pk_stats_30bin', stats_30bin)

print(f'\nchains left in place, at {chain_dir}')

  /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains/chain_convergence.txt

chains left in place, at /gpfs/projects/SehgalGroup/zcheslog/cmb_hd/hdInitPk_upload_latest/hdinitpk/data/user_generated_data/chains


## Using these in the plotting notebook

At the top of `hdInitPk_plots.ipynb`, set

```python
chain_source = 'custom'
```

and the figures and tables will be built from your chains instead of the
thinned ones distributed with the package. Everything else in that notebook
is unchanged. The Fisher forecasts, the Figure 6 points and the binning
files still come from `hdinitpk/data`.

The plotting notebook loads all five power-law chains from the same place
and removes the burn-in from each on load, so they should all be your own.
The binned $\mathcal{P}(k)$ statistics are only read from the cached files,
so for a binned run you did not repeat you can copy the packaged file from
`hdinitpk/data/cached_results` into your own `cached_results` directory.